<a href="https://colab.research.google.com/github/ravzaoguz/Machine-Learning/blob/main/airlab%C3%B6devson.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import confusion_matrix
import seaborn as sns
import numpy as np
import random

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
trainset = torchvision.datasets.FashionMNIST(
    root="./data",
    train=True,
    download=True,
    transform=transforms.ToTensor()
)

testset = torchvision.datasets.FashionMNIST(
    root="./data",
    train=False,
    download=True,
    transform=transforms.ToTensor()
)

train_loader = DataLoader(
    trainset,
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    testset,
    batch_size=64,
    shuffle=False
)

print("eğitim örnek sayısı:", len(trainset))
print("test örnek sayısı:", len(testset))

images, labels = next(iter(train_loader))

print("images şekli:", images.shape)
print("labels şekli:", labels.shape)



class_names = [
    "T-shirt", "Pantolon", "Kazak", "Elbise", "Mont",
    "Sandalet", "Gömlek", "Spor Ayakkabı", "Çanta", "Bot"
]

plt.imshow(images[0].squeeze(), cmap="gray")
plt.title(class_names[labels[0].item()])
plt.axis("off")
plt.show()


In [ ]:
class SimpleCNN(nn.Module):

    def __init__(self):

        super(SimpleCNN, self).__init__()

        self.conv1 = nn.Conv2d(
            in_channels=1,
            out_channels=16,
            kernel_size=3,
            padding=1
        )

        self.pool = nn.MaxPool2d(
            kernel_size=2,
            stride=2
        )

        self.conv2 = nn.Conv2d(
            in_channels=16,
            out_channels=32,
            kernel_size=3,
            padding=1
        )

        self.fc1 = nn.Linear(
            in_features=32*7*7,
            out_features=128
        )

        self.fc2 = nn.Linear(
            in_features=128,
            out_features=10
        )

    def forward(self, x):

        x = self.pool(torch.relu(self.conv1(x)))

        x = self.pool(torch.relu(self.conv2(x)))

        x = torch.flatten(x, 1)

        x = torch.relu(self.fc1(x))

        x = self.fc2(x)

        return x


model = SimpleCNN()

print(model)



device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)


In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

epochs = 3

print("Eğitim başlıyor.")

for epoch in range(epochs):

    toplam_hata = 0.0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        tahminler = model(images)

        loss = criterion(tahminler, labels)

        loss.backward()

        optimizer.step()

        toplam_hata += loss.item()

    ortalama_hata = toplam_hata / len(train_loader)

    print(
        f"tur (Epoch) {epoch+1} tamamlandı. "
        f"ortalama hata (loss): {ortalama_hata:.4f}"
    )

print("Eğitim bitti.")



In [ ]:
def model_degerlendir(model, loader):

    model.eval()

    dogru_tahmin = 0
    toplam_soru = 0

    tum_gercekler = []
    tum_tahminler = []

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            predicted = outputs.argmax(dim=1)

            toplam_soru += labels.size(0)

            dogru_tahmin += (
                predicted == labels
            ).sum().item()

            tum_gercekler.extend(labels.cpu().numpy())
            tum_tahminler.extend(predicted.cpu().numpy())

    basari_orani = 100 * dogru_tahmin / toplam_soru

    return basari_orani, tum_gercekler, tum_tahminler



basari_orani, gercekler_temiz, tahminler_temiz = model_degerlendir(
    model,
    test_loader
)

baseline_temiz_skor = basari_orani

print(f"Testin doğruluk oranı: %{basari_orani:.2f}")



In [ ]:
kural_hafif_blur = transforms.Compose([
    transforms.GaussianBlur(kernel_size=3, sigma=0.5),
    transforms.ToTensor()
])

kural_orta_blur = transforms.Compose([
    transforms.GaussianBlur(kernel_size=5, sigma=1.0),
    transforms.ToTensor()
])

kural_agir_blur = transforms.Compose([
    transforms.GaussianBlur(kernel_size=7, sigma=2.0),
    transforms.ToTensor()
])


testset_hafif = torchvision.datasets.FashionMNIST(
    root='./data',
    train=False,
    download=True,
    transform=kural_hafif_blur
)

testset_orta = torchvision.datasets.FashionMNIST(
    root='./data',
    train=False,
    download=True,
    transform=kural_orta_blur
)

testset_agir = torchvision.datasets.FashionMNIST(
    root='./data',
    train=False,
    download=True,
    transform=kural_agir_blur
)


loader_hafif = DataLoader(
    testset_hafif,
    batch_size=64,
    shuffle=False
)

loader_orta = DataLoader(
    testset_orta,
    batch_size=64,
    shuffle=False
)

loader_agir = DataLoader(
    testset_agir,
    batch_size=64,
    shuffle=False
)

class GaussianNoiseDataset(Dataset):

    def __init__(self, dataset, sigma, seed=42):

        self.dataset = dataset
        self.sigma = sigma
        self.seed = seed

    def __len__(self):

        return len(self.dataset)

    def __getitem__(self, index):

        image, label = self.dataset[index]

        generator = torch.Generator()
        generator.manual_seed(self.seed + index)

        noise = torch.randn(
            image.shape,
            generator=generator
        ) * self.sigma

        image = torch.clamp(image + noise, 0, 1)

        return image, label


testset_noise_hafif = GaussianNoiseDataset(
    testset, sigma=0.05
)

testset_noise_orta = GaussianNoiseDataset(
    testset, sigma=0.10
)

testset_noise_agir = GaussianNoiseDataset(
    testset, sigma=0.20
)


loader_noise_hafif = DataLoader(
    testset_noise_hafif,
    batch_size=64,
    shuffle=False
)

loader_noise_orta = DataLoader(
    testset_noise_orta,
    batch_size=64,
    shuffle=False
)

loader_noise_agir = DataLoader(
    testset_noise_agir,
    batch_size=64,
    shuffle=False
)


kural_hafif_parlaklik = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(
        lambda img: transforms.functional.adjust_brightness(img, 0.8)
    )
])

kural_orta_parlaklik = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(
        lambda img: transforms.functional.adjust_brightness(img, 0.6)
    )
])

kural_agir_parlaklik = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(
        lambda img: transforms.functional.adjust_brightness(img, 0.4)
    )
])


testset_parlaklik_hafif = torchvision.datasets.FashionMNIST(
    root='./data',
    train=False,
    download=True,
    transform=kural_hafif_parlaklik
)

testset_parlaklik_orta = torchvision.datasets.FashionMNIST(
    root='./data',
    train=False,
    download=True,
    transform=kural_orta_parlaklik
)

testset_parlaklik_agir = torchvision.datasets.FashionMNIST(
    root='./data',
    train=False,
    download=True,
    transform=kural_agir_parlaklik
)


loader_parlaklik_hafif = DataLoader(
    testset_parlaklik_hafif,
    batch_size=64,
    shuffle=False
)

loader_parlaklik_orta = DataLoader(
    testset_parlaklik_orta,
    batch_size=64,
    shuffle=False
)

loader_parlaklik_agir = DataLoader(
    testset_parlaklik_agir,
    batch_size=64,
    shuffle=False
)



kural_hafif_rotation = transforms.Compose([
    transforms.Lambda(
        lambda img: transforms.functional.rotate(img, 5)
    ),
    transforms.ToTensor()
])

kural_orta_rotation = transforms.Compose([
    transforms.Lambda(
        lambda img: transforms.functional.rotate(img, 15)
    ),
    transforms.ToTensor()
])

kural_agir_rotation = transforms.Compose([
    transforms.Lambda(
        lambda img: transforms.functional.rotate(img, 30)
    ),
    transforms.ToTensor()
])


testset_rotation_hafif = torchvision.datasets.FashionMNIST(
    root='./data',
    train=False,
    download=True,
    transform=kural_hafif_rotation
)

testset_rotation_orta = torchvision.datasets.FashionMNIST(
    root='./data',
    train=False,
    download=True,
    transform=kural_orta_rotation
)

testset_rotation_agir = torchvision.datasets.FashionMNIST(
    root='./data',
    train=False,
    download=True,
    transform=kural_agir_rotation
)


loader_rotation_hafif = DataLoader(
    testset_rotation_hafif,
    batch_size=64,
    shuffle=False
)

loader_rotation_orta = DataLoader(
    testset_rotation_orta,
    batch_size=64,
    shuffle=False
)

loader_rotation_agir = DataLoader(
    testset_rotation_agir,
    batch_size=64,
    shuffle=False
)


In [ ]:
test_loaderlari = {

    "Temiz Veri": test_loader,

    "Hafif Bulanıklık": loader_hafif,
    "Orta Bulanıklık": loader_orta,
    "Ağır Bulanıklık": loader_agir,

    "Hafif Gürültü": loader_noise_hafif,
    "Orta Gürültü": loader_noise_orta,
    "Ağır Gürültü": loader_noise_agir,

    "Hafif Parlaklık": loader_parlaklik_hafif,
    "Orta Parlaklık": loader_parlaklik_orta,
    "Ağır Parlaklık": loader_parlaklik_agir,

    "Hafif Döndürme": loader_rotation_hafif,
    "Orta Döndürme": loader_rotation_orta,
    "Ağır Döndürme": loader_rotation_agir
}


baseline_sonuclari = {}
baseline_tahminleri = {}

print("\n--- BASELINE MODEL TEST SONUÇLARI ---")

for isim, loader in test_loaderlari.items():

    skor, gercekler, tahminler = model_degerlendir(
        model,
        loader
    )

    baseline_sonuclari[isim] = skor

    baseline_tahminleri[isim] = (
        gercekler,
        tahminler
    )

    print(f"{isim}: %{skor:.2f}")


baseline_agir_skor = baseline_sonuclari["Ağır Bulanıklık"]



In [ ]:
kiyafet_isimleri = [
    'T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]


def hata_matrisi(gercekler, tahminler, baslik):

    matris = confusion_matrix(
        gercekler,
        tahminler,
        labels=list(range(10))
    )

    plt.figure(figsize=(10, 8))

    sns.heatmap(
        matris,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=kiyafet_isimleri,
        yticklabels=kiyafet_isimleri
    )

    plt.title(baslik)
    plt.ylabel('GERÇEK KIYAFET')
    plt.xlabel('MODELİN TAHMİNİ')

    plt.tight_layout()
    plt.show()
    gercekler, tahminler = baseline_tahminleri["Temiz Veri"]

hata_matrisi(
    gercekler,
    tahminler,
    "Hata Matrisi - Temiz Veri (Baseline)"
)


gercekler, tahminler = baseline_tahminleri["Ağır Bulanıklık"]

hata_matrisi(
    gercekler,
    tahminler,
    "Hata Matrisi - Ağır Bulanıklık (Baseline)"
)


In [ ]:
def sinif_bazli_basari(gercekler, tahminler):

    matris = confusion_matrix(
        gercekler,
        tahminler,
        labels=list(range(10))
    )

    print("\n--- SINIF BAZLI BAŞARI ---")

    for i in range(10):

        toplam = matris[i].sum()
        dogru = matris[i, i]

        basari = 100 * dogru / toplam if toplam > 0 else 0

        print(
            f"{kiyafet_isimleri[i]}: %{basari:.2f}"
        )


gercekler, tahminler = baseline_tahminleri["Temiz Veri"]

sinif_bazli_basari(gercekler, tahminler)


gercekler, tahminler = baseline_tahminleri["Ağır Bulanıklık"]

sinif_bazli_basari(gercekler, tahminler)



In [ ]:
lass AddGaussianNoise:

    def __init__(self, sigma=0.08):

        self.sigma = sigma

    def __call__(self, image):

        noise = torch.randn(image.size()) * self.sigma

        return torch.clamp(image + noise, 0, 1)


kural_aug = transforms.Compose([

    transforms.RandomApply(
        [transforms.GaussianBlur(3, sigma=(0.5, 1.5))],
        p=0.4
    ),

    transforms.RandomApply(
        [transforms.ColorJitter(brightness=0.3, contrast=0.3)],
        p=0.3
    ),

    transforms.RandomRotation(degrees=10),

    transforms.ToTensor(),

    transforms.RandomApply(
        [AddGaussianNoise(sigma=0.08)],
        p=0.3
    )

])


trainset_aug = torchvision.datasets.FashionMNIST(
    root='./data',
    train=True,
    download=True,
    transform=kural_aug
)

trainloader_aug = DataLoader(
    trainset_aug,
    batch_size=64,
    shuffle=True
)
model_robust = SimpleCNN().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model_robust.parameters(),
    lr=0.001
)


print("\nAşılı Model Eğitiliyor (Data Augmentation)")


for epoch in range(3):

    toplam_hata = 0.0

    for images, labels in trainloader_aug:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        tahminler = model_robust(images)

        loss = criterion(tahminler, labels)

        loss.backward()

        optimizer.step()

        toplam_hata += loss.item()

    ortalama_hata = toplam_hata / len(trainloader_aug)

    print(
        f"tur (Epoch) {epoch+1} tamamlandı. "
        f"ortalama hata (loss): {ortalama_hata:.4f}"
    )


print("Aşılı Modelin Eğitimi Bitti.")



In [ ]:
robust_sonuclari = {}
robust_tahminleri = {}

print("\nAŞILI MODEL TEST SONUÇLARI")


for isim, loader in test_loaderlari.items():

    skor, gercekler, tahminler = model_degerlendir(
        model_robust,
        loader
    )

    robust_sonuclari[isim] = skor

    robust_tahminleri[isim] = (
        gercekler,
        tahminler
    )

    print(f"{isim}: %{skor:.2f}")
gercekler, tahminler = robust_tahminleri["Ağır Bulanıklık"]

hata_matrisi(
    gercekler,
    tahminler,
    "Hata Matrisi - Ağır Bulanıklık (Aşılı Model)"
)

print("\n========== TRADE-OFF RAPORU ==========")

print("\n1. İYİLEŞME (ROBUSTNESS)")

for isim in test_loaderlari:

    eski_skor = baseline_sonuclari[isim]
    yeni_skor = robust_sonuclari[isim]

    fark = yeni_skor - eski_skor

    print(
        f"{isim}: "
        f"Eski Model: %{eski_skor:.2f} | "
        f"Aşılı Model: %{yeni_skor:.2f} | "
        f"Değişim: {fark:+.2f} yüzde puan"
    )


print("\n2. TEMİZ VERİ PERFORMANSI")

baseline_temiz_skor = baseline_sonuclari["Temiz Veri"]

yeni_temiz_skor = robust_sonuclari["Temiz Veri"]

print(
    f"Eski Model Temiz Veri Başarısı: "
    f"%{baseline_temiz_skor:.2f}"
)

print(
    f"Aşılı Model Temiz Veri Başarısı: "
    f"%{yeni_temiz_skor:.2f}"
)


temiz_veri_farki = yeni_temiz_skor - baseline_temiz_skor

print(
    f"Temiz veri performans değişimi: "
    f"{temiz_veri_farki:+.2f} yüzde puan"
)


print("\n3. AĞIR BULANIKLIK PERFORMANSI")

baseline_agir_skor = baseline_sonuclari["Ağır Bulanıklık"]

yeni_agir_skor = robust_sonuclari["Ağır Bulanıklık"]

print(
    f"Eski Model Ağır Bulanıklık Başarısı: "
    f"%{baseline_agir_skor:.2f}"
)

print(
    f"Aşılı Model Ağır Bulanıklık Başarısı: "
    f"%{yeni_agir_skor:.2f}"
)


agir_veri_farki = yeni_agir_skor - baseline_agir_skor

print(
    f"Ağır bulanıklık performans değişimi: "
    f"{agir_veri_farki:+.2f} yüzde puan"
)



In [ ]:
isimler = list(baseline_sonuclari.keys())

eski_skorlar = list(baseline_sonuclari.values())

yeni_skorlar = list(robust_sonuclari.values())

x = np.arange(len(isimler))

genislik = 0.35

plt.figure(figsize=(16, 7))

plt.bar(
    x - genislik/2,
    eski_skorlar,
    genislik,
    label="Baseline Model"
)

plt.bar(
    x + genislik/2,
    yeni_skorlar,
    genislik,
    label="Aşılı Model"
)

plt.xticks(
    x,
    isimler,
    rotation=45,
    ha="right"
)

plt.ylabel("Doğruluk (%)")

plt.title("Baseline ve Aşılı Model Performans Karşılaştırması")

plt.legend()

plt.tight_layout()

plt.show()



In [ ]:
degisimler = [
    robust_sonuclari[isim] - baseline_sonuclari[isim]
    for isim in isimler
]

plt.figure(figsize=(14, 6))

plt.bar(isimler, degisimler)

plt.axhline(
    y=0,
    linestyle="--"
)

plt.xticks(
    rotation=45,
    ha="right"
)

plt.ylabel("Doğruluk Değişimi (Yüzde Puan)")

plt.title("Data Augmentation Sonrası Performans Değişimi")

plt.tight_layout()

plt.show()


print("\nDeney tamamlandı.")